## In this document, I combine the  vectorized data  using BERT with the 41,000-row data I already had and check for similarities.

#### PS: The combination worked, but the vectors in one were 2D and in the other 1D, or there was another problem, in short, it was a size-related problem. For this, we will use the data separately.

In [23]:
import pickle

# Load the pickle file
with open("embeddings.pkl", "rb") as file:
    data = pickle.load(file)

# Print the type of the loaded object
print(f"Type of object: {type(data)}")

# Print a preview of the content
if isinstance(data, dict):
    print("Keys:", list(data.keys())[:10])  # Show first 10 keys
elif isinstance(data, list):
    print("First 5 elements:", data[:5])  # Show first 5 elements
else:
    print("Content:", data)


Type of object: <class 'numpy.ndarray'>
Content: [[ 0.06643411 -0.04954599  0.06388086 ...  0.00106303 -0.12156384
  -0.06962775]
 [ 0.09212254 -0.07606936  0.06572869 ... -0.08565165 -0.09266549
   0.00725293]
 [-0.08162683  0.02428931  0.01888745 ...  0.00806158 -0.05129531
  -0.05873992]
 ...
 [ 0.01227977 -0.08568829 -0.02782772 ... -0.05257971 -0.10806689
   0.07843318]
 [-0.07258204 -0.12690921 -0.00535557 ...  0.03597704 -0.03986148
  -0.05971025]
 [ 0.00768868 -0.1012418   0.08909854 ... -0.08199869 -0.05649744
   0.09007054]]


In [6]:
data.shape

(41105, 384)

In [7]:
data[:5]    # Preview first 5 feature vectors


array([[ 0.06643411, -0.04954599,  0.06388086, ...,  0.00106303,
        -0.12156384, -0.06962775],
       [ 0.09212254, -0.07606936,  0.06572869, ..., -0.08565165,
        -0.09266549,  0.00725293],
       [-0.08162683,  0.02428931,  0.01888745, ...,  0.00806158,
        -0.05129531, -0.05873992],
       [-0.13487624, -0.04894156,  0.03343301, ...,  0.03911195,
        -0.02554824, -0.07314679],
       [-0.00808762, -0.04798989,  0.03134545, ...,  0.11500218,
         0.0255291 , -0.01095645]], dtype=float32)

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(data)
print(similarity_matrix[:5, :5])

[[1.0000002  0.24351445 0.4486773  0.35114962 0.24552569]
 [0.24351445 1.0000002  0.13602321 0.12184064 0.21296859]
 [0.4486773  0.13602321 1.0000004  0.46773392 0.1195616 ]
 [0.35114962 0.12184064 0.46773392 1.0000001  0.10363281]
 [0.24552569 0.21296859 0.1195616  0.10363281 1.0000002 ]]


In [9]:
import numpy as np

# Exclude self-similarity by setting diagonal to zero
np.fill_diagonal(similarity_matrix, 0)

# Find most similar pairs
most_similar = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
print(f"Most similar documents: {most_similar}")
print(f"Similarity score: {similarity_matrix[most_similar]}")


Most similar documents: (6990, 11135)
Similarity score: 1.000000238418579


In [10]:
plagiarized_pairs = np.argwhere(similarity_matrix > 0.8)
print(plagiarized_pairs)


[[    2 39757]
 [    5    93]
 [    9   488]
 ...
 [41100 39869]
 [41101 22385]
 [41101 37401]]


## checking if given sentence is plagiarised

In [53]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load stored embeddings
with open("embeddings.pkl", "rb") as file:
    doc_embeddings = pickle.load(file)

# Load the original text dataset (if available)
with open("sentences.pkl", "rb") as file:
    original_texts = pickle.load(file)  # Must contain the original text of each document

# Define the vectorizer (same one used to generate embeddings)
vectorizer = TfidfVectorizer()

# Fit vectorizer on the original documents and transform them
vectorizer.fit(original_texts)
doc_embeddings = vectorizer.transform(original_texts)

# Transform the new input sentence
new_sentence = "Machine learning is a subfield of AI."
new_embedding = vectorizer.transform([new_sentence])

# Compute similarity
similarities = cosine_similarity(new_embedding, doc_embeddings)

# Find most similar document
max_similarity = similarities.max()
most_similar_index = similarities.argmax()

print(f"Most similar document: {original_texts[most_similar_index]}")
print(f"Similarity score: {max_similarity}")


Most similar document: How Personal is Machine Learning Personalization?
Similarity score: 0.36968382274045675


In [19]:
original_texts.info

<bound method Series.info of 0        Multi-Level Attention Pooling for Graph Neural...
1        Decision Forests vs. Deep Networks: Conceptual...
2        Power up! Robust Graph Convolutional Network v...
3        Releasing Graph Neural Networks with Different...
4        Recurrence-Aware Long-Term Cognitive Network f...
                               ...                        
41100    An experimental study of graph-based semi-supe...
41101    Bayesian Differential Privacy through Posterio...
41102    Mining Spatio-temporal Data on Industrializati...
41103    Wav2Letter: an End-to-End ConvNet-based Speech...
41104                          Generalized Low Rank Models
Name: titles, Length: 41105, dtype: object>

## Concating all vectorised datasets

In [26]:
with open("embeddingsSTEM.pkl", "rb") as f:
    data0 = pickle.load(f)
data0

,text,url,title,vectors
0,The Ulakhan Fault is a left-lateral moving tra...,https://en.wikipedia.org/wiki/Ulakhan%20Fault,Ulakhan Fault,"[[-0.69774044, -0.117761, -0.3979759, 0.027350..."
1,Tripodal ligands are tri- and tetradentate lig...,https://en.wikipedia.org/wiki/Tripodal%20ligand,Tripodal ligand,"[[-0.43669507, 0.02436654, -0.50654644, -0.881..."
2,WASP-14b is an extrasolar planet discovered in...,https://en.wikipedia.org/wiki/WASP-14b,WASP-14b,"[[-0.8890304, 0.19592004, -0.16842899, -0.0435..."
3,Long-acting reversible contraceptives (LARC) a...,https://en.wikipedia.org/wiki/Long-acting%20re...,Long-acting reversible contraception,"[[-0.6982364, -0.023231473, -0.35311332, 0.138..."
4,"AIDS (""acquired immune deficiency syndrome"") i...",https://en.wikipedia.org/wiki/HIV%20integration,HIV integration,"[[-0.3878806, -0.11851628, -0.46014228, -0.304..."
...,...,...,...,...
91723,Phenyl phosphorodiamidate is an organophosphor...,https://en.wikipedia.org/wiki/Phenyl%20phospho...,Phenyl phosphorodiamidate,"[[-0.3785945, 0.16301002, -0.12910874, -0.3632..."
91724,"Yariv reagent (1,3,5-tri(p-glycosyloxyphenylaz...",https://en.wikipedia.org/wiki/Yariv%20reagent,Yariv reagent,"[[-0.55147815, -0.038644735, -0.39247343, -0.0..."
91725,Thaspiviridae is a family of spindle-shaped vi...,https://en.wikipedia.org/wiki/Thaspiviridae,Thaspiviridae,"[[0.39006183, 0.123846985, -0.70239383, -0.035..."
91726,Rankine is the codename for a GPU microarchite...,https://en.wikipedia.org/wiki/Rankine%20%28mic...,Rankine (microarchitecture),"[[-0.6339383, -0.2796811, -0.0019871802, 0.241..."


In [24]:
# Load stored embeddings
with open("embeddings.pkl", "rb") as file:
    doc_embeddings = pickle.load(file)

# Load the original text dataset (if available)
with open("sentences.pkl", "rb") as file:
    original_texts = pickle.load(file)  # Must contain the original text of each document
original_texts

0        Multi-Level Attention Pooling for Graph Neural...
1        Decision Forests vs. Deep Networks: Conceptual...
2        Power up! Robust Graph Convolutional Network v...
3        Releasing Graph Neural Networks with Different...
4        Recurrence-Aware Long-Term Cognitive Network f...
                               ...                        
41100    An experimental study of graph-based semi-supe...
41101    Bayesian Differential Privacy through Posterio...
41102    Mining Spatio-temporal Data on Industrializati...
41103    Wav2Letter: an End-to-End ConvNet-based Speech...
41104                          Generalized Low Rank Models
Name: titles, Length: 41105, dtype: object

In [27]:
import pandas as pd
assert len(original_texts) == len(doc_embeddings), "Mismatch in text and embedding sizes!"

# Convert NumPy array to a list for DataFrame compatibility
doc_embeddings_list = [emb.reshape(1, -1) for emb in doc_embeddings]

# Create a new DataFrame with the required columns
new_df = pd.DataFrame({
    "text": original_texts,  # Original texts
    "url": [""] * len(original_texts),  # Placeholder URL (if missing)
    "title": original_texts,  # Assuming titles are the same as text
    "vectors": doc_embeddings_list  # Embeddings as list
})


# Append the new data to final_df
final_df = pd.concat([data0, new_df], ignore_index=True)

# Save the updated DataFrame
with open("final_embeddings.pkl", "wb") as f:
    pickle.dump(final_df, f)

print("✅ Merging complete! Saved as 'final_embeddings.pkl'.")

✅ Merging complete! Saved as 'final_embeddings.pkl'.


In [11]:
from IPython.display import FileLink
FileLink("final_embeddings.pkl")

C:\Users\ADMIN\plagiarism\final_embeddings.pkl

In [30]:
final_df.tail()

,text,url,title,vectors
132828,An experimental study of graph-based semi-supe...,,An experimental study of graph-based semi-supe...,"[[-0.009588274, 0.011530031, 0.032066435, 0.01..."
132829,Bayesian Differential Privacy through Posterio...,,Bayesian Differential Privacy through Posterio...,"[[-0.11083766, -0.053484198, 0.0062898695, -0...."
132830,Mining Spatio-temporal Data on Industrializati...,,Mining Spatio-temporal Data on Industrializati...,"[[0.012279772, -0.08568829, -0.027827717, -0.0..."
132831,Wav2Letter: an End-to-End ConvNet-based Speech...,,Wav2Letter: an End-to-End ConvNet-based Speech...,"[[-0.07258204, -0.12690921, -0.005355567, -0.0..."
132832,Generalized Low Rank Models,,Generalized Low Rank Models,"[[0.007688676, -0.101241805, 0.08909854, 0.040..."


In [39]:
import numpy as np

def flatten_vectors(vectors):
    flattened = []
    for i, vector in enumerate(vectors):
        # Print only the first 10 vectors
        if i < 10:
            print(f"Original vector shape (index {i}): {np.array(vector).shape}")
            if np.array(vector).ndim == 3:
                flattened.append(np.array(vector).reshape(-1))
            else:
                flattened.append(np.array(vector).flatten())
            print(f"Flattened vector shape (index {i}): {np.array(flattened[-1]).shape}")
        else:
            flattened.append(np.array(vector).flatten())
    return flattened

# Assuming final_df['vectors'] is your input column
final_df['vectors'] = flatten_vectors(final_df['vectors'])


Original vector shape (index 0): (768,)
Flattened vector shape (index 0): (768,)
Original vector shape (index 1): (768,)
Flattened vector shape (index 1): (768,)
Original vector shape (index 2): (768,)
Flattened vector shape (index 2): (768,)
Original vector shape (index 3): (768,)
Flattened vector shape (index 3): (768,)
Original vector shape (index 4): (768,)
Flattened vector shape (index 4): (768,)
Original vector shape (index 5): (768,)
Flattened vector shape (index 5): (768,)
Original vector shape (index 6): (768,)
Flattened vector shape (index 6): (768,)
Original vector shape (index 7): (768,)
Flattened vector shape (index 7): (768,)
Original vector shape (index 8): (768,)
Flattened vector shape (index 8): (768,)
Original vector shape (index 9): (768,)
Flattened vector shape (index 9): (768,)


In [37]:
def check_vector_shapes(vectors):
    shapes = []
    for vector in vectors:
        shapes.append(np.array(vector).shape)
    return shapes

# Check the shapes of all vectors
vector_shapes = check_vector_shapes(final_df['vectors'])

## Runing the plagiarism checker model

In [2]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load the final dataset
with open("final_embeddings.pkl", "rb") as file:
    final_df = pickle.load(file)

# Extract text column for vectorization
text_corpus = final_df["text"].tolist()

# Define the vectorizer and fit on dataset
vectorizer = TfidfVectorizer()
vectorizer.fit(text_corpus)
doc_embeddings = vectorizer.transform(text_corpus)

# New input sentence to check for plagiarism
new_sentence = "Principal components analysis (PCA) is a well-known technique for approximating a tabular data set by a low rank matrix. Here, we extend the idea of PCA to handle arbitrary data sets consisting of numerical, Boolean, categorical, ordinal, and other data types. This framework encompasses many well known techniques in data analysis, such as nonnegative matrix factorization, matrix completion, sparse and robust PCA, k-means, k-SVD, and maximum margin matrix factorization. The method handles heterogeneous data sets, and leads to coherent schemes for compressing, denoising, and imputing missing entries across all data types simultaneously. It also admits a number of interesting interpretations of the low rank factors, which allow clustering of examples or of features. We propose several parallel algorithms for fitting generalized low rank models, and describe implementations and numerical results."

# Transform the new sentence
new_embedding = vectorizer.transform([new_sentence])

# Compute cosine similarity
similarities = cosine_similarity(new_embedding, doc_embeddings)

# Find the most similar document
max_similarity = similarities.max()
most_similar_index = similarities.argmax()

print("🔎 Plagiarism Detection Result:")
print(f"Most similar document: {final_df.iloc[most_similar_index]['text']}")
print(f"URL: {final_df.iloc[most_similar_index]['url']}")
print(f"Title: {final_df.iloc[most_similar_index]['title']}")
print(f"Similarity Score: {max_similarity:.4f}")

# Define a threshold for potential plagiarism (e.g., 0.75 means 75% similarity)
PLAGIARISM_THRESHOLD = 0.75
if max_similarity >= PLAGIARISM_THRESHOLD:
    print("🚨 Possible plagiarism detected!")
else:
    print("✅ No plagiarism detected.")

🔎 Plagiarism Detection Result:
Most similar document: Robust Principal Component Analysis (RPCA) is a modification of the widely used statistical procedure of principal component analysis (PCA) which works well with respect to grossly corrupted observations. A number of different approaches exist for Robust PCA, including an idealized version of Robust PCA, which aims to recover a low-rank matrix L0 from highly corrupted measurements M = L0 +S0. This decomposition in low-rank and sparse matrices can be achieved by techniques such as Principal Component Pursuit method (PCP), Stable PCP, Quantized PCP, Block based PCP, and Local PCP. Then, optimization methods are used such as the Augmented Lagrange Multiplier Method (ALM), Alternating Direction Method (ADM), Fast Alternating Minimization (FAM), Iteratively Reweighted Least Squares (IRLS ) 
or alternating projections (AP).

Algorithms

Non-convex method
The 2014 guaranteed algorithm for the robust PCA problem (with the input matrix being

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
import pickle

# Load your precomputed embeddings
with open("embeddingsSTEM.pkl", "rb") as file:
    data0 = pickle.load(file)

data0_texts = data0['text']  # The texts associated with the embeddings

# Function to compute Jaccard similarity
def jaccard_similarity(str1, str2):
    # Convert texts into sets of words
    set1 = set(str1.split())
    set2 = set(str2.split())
    
    # Return the Jaccard similarity between two sets
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    return len(intersection) / len(union)

def check_local_plagiarism(new_sentence):
    similarities = []

    for text in data0_texts:
        similarities.append(jaccard_similarity(new_sentence, text))

    # Find the most similar document
    max_similarity = max(similarities)
    most_similar_index = similarities.index(max_similarity)

    return data0_texts[most_similar_index], max_similarity

# Test the function
new_sentence = input("Enter the text to check for plagiarism: ")

local_text, similarity_score = check_local_plagiarism(new_sentence)

print(f"Most similar document: {local_text}")
print(f"Similarity score: {similarity_score}")


Enter the text to check for plagiarism: The Ulakhan Fault is a left-lateral transform fault that runs along the boundary between the North American plate and the Okhotsk microplate in northeast Asia.
Most similar document: The Ulakhan Fault is a left-lateral moving transform fault that runs along the boundary between two tectonic plates in northeast Asia, the North American Plate, and the Okhotsk Plate.  It runs from a triple junction in the Chersky Range in the west, to another triple junction with the Aleutian Trench and the Kuril Trench in the east. From the offset of dated geomorphological features, a slip rate of 5.3±1.3 mm per year has been measured, consistent with estimates from GPS constrained global plate models. The analysis of fault scarps along the fault zone in the Seymchan Basin suggests that the fault is characterised by occasional large (>7.5) earthquakes.

References

External links
 International Seismic-Volcanic Workshop on Kamchatkan-Aleutian Subduction Processes (

In [2]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ADMIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


## BERT model -this works so well 20.02.2025 19.56

In [8]:
!pip -q install transformers
import torch
from keras.preprocessing.sequence import pad_sequences
from transformers import BertTokenizer,  AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoModel

# Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
import pandas as pd
import numpy as np
import ast

# Clean the 'vectors' column (only process string values)
def clean_data(text):
    if isinstance(text, str):  # Check if the element is a string
        return text.replace("\0", "")  # Remove null bytes if any
    return text  # If it's already an array, just return it as is

# Apply the cleaning function to remove null bytes
df['vectors'] = df['vectors'].apply(clean_data)  # Clean the data

# Now safely convert string representation of vectors to numpy arrays
def safe_eval(x):
    if isinstance(x, str):  # Only attempt to eval if it's a string
        try:
            return np.array(ast.literal_eval(x))  # Convert to numpy array
        except Exception as e:
            print(f"Error converting vector: {e}")
            return np.array([])  # Return empty array if eval fails
    return x  # If it's already an array, just return it as is

# Apply safe_eval to the 'vectors' column
df['vectors'] = df['vectors'].apply(safe_eval)

# Now 'df['vectors']' should contain numpy arrays without errors


NameError: name 'df' is not defined

In [2]:
model_path = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_path, 
                                          do_lower_case=True)

model = AutoModelForSequenceClassification.from_pretrained(model_path,
                                                          output_attentions=False,
                                                          output_hidden_states=True)

model.safetensors:  24%|##3       | 105M/440M [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/bert-base-uncased/68d45e234eb4a928074dfd868cead0219ab85354cc53d20e772753c6bb9169d3?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1740063347&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MDA2MzM0N319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9iZXJ0LWJhc2UtdW5jYXNlZC82OGQ0NWUyMzRlYjRhOTI4MDc0ZGZkODY4Y2VhZDAyMTlhYjg1MzU0Y2M1M2QyMGU3NzI3NTNjNmJiOTE2OWQzP3Jlc3BvbnNlLWNvbnRlbnQtZGlzcG9zaXRpb249KiJ9XX0_&Signature=ED8eCodE9KHgLycSqmi8rHEk8E0wHz3V6%7ECppCzTtHbXHlMKAL4bppxaAILuycD5FwaMpxWQnz7V57Xk6SLe9hm5Ig4Wuk93ESo4XOW0JFmxW-RRW7eMOjdJjX6TzuuFVA7UJEHE%7EvT5J4SD5YbTVcBqV03BEkeRYoGWRyqjdGOk0iG%7EcS7z799yQYJWMbqadQBPHYakVo5m7-DbxQenvSfAWgZMfagoSE%7EtyLRSCbzaW0UXtaQ5V%7EDIG6w3DfxuExolZ6DV1ZwgU8100BCF1f46P%7Ep3vhV-pIcIPP1nuXXca%7E%7EL7%7ExuBjEToKeXyFxGQPVFM10I5sMGmbgtgPpjNQ__&Key-Pair-Id=K3RPWS32NSSJCE: HTTPSConnectionPo

model.safetensors:  38%|###8      | 168M/440M [00:00<?, ?B/s]

C:\Users\ADMIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly

In [18]:
import pickle
with open("embeddingsSTEM.pkl", "rb") as file:
    data = pickle.load(file)
data

,text,url,title,vectors
0,The Ulakhan Fault is a left-lateral moving tra...,https://en.wikipedia.org/wiki/Ulakhan%20Fault,Ulakhan Fault,"[[-0.69774044, -0.117761, -0.3979759, 0.027350..."
1,Tripodal ligands are tri- and tetradentate lig...,https://en.wikipedia.org/wiki/Tripodal%20ligand,Tripodal ligand,"[[-0.43669507, 0.02436654, -0.50654644, -0.881..."
2,WASP-14b is an extrasolar planet discovered in...,https://en.wikipedia.org/wiki/WASP-14b,WASP-14b,"[[-0.8890304, 0.19592004, -0.16842899, -0.0435..."
3,Long-acting reversible contraceptives (LARC) a...,https://en.wikipedia.org/wiki/Long-acting%20re...,Long-acting reversible contraception,"[[-0.6982364, -0.023231473, -0.35311332, 0.138..."
4,"AIDS (""acquired immune deficiency syndrome"") i...",https://en.wikipedia.org/wiki/HIV%20integration,HIV integration,"[[-0.3878806, -0.11851628, -0.46014228, -0.304..."
...,...,...,...,...
91723,Phenyl phosphorodiamidate is an organophosphor...,https://en.wikipedia.org/wiki/Phenyl%20phospho...,Phenyl phosphorodiamidate,"[[-0.3785945, 0.16301002, -0.12910874, -0.3632..."
91724,"Yariv reagent (1,3,5-tri(p-glycosyloxyphenylaz...",https://en.wikipedia.org/wiki/Yariv%20reagent,Yariv reagent,"[[-0.55147815, -0.038644735, -0.39247343, -0.0..."
91725,Thaspiviridae is a family of spindle-shaped vi...,https://en.wikipedia.org/wiki/Thaspiviridae,Thaspiviridae,"[[0.39006183, 0.123846985, -0.70239383, -0.035..."
91726,Rankine is the codename for a GPU microarchite...,https://en.wikipedia.org/wiki/Rankine%20%28mic...,Rankine (microarchitecture),"[[-0.6339383, -0.2796811, -0.0019871802, 0.241..."


In [6]:
def create_vector_from_text(tokenizer, model, text, MAX_LEN=512):
    # Tokenize and pad input
    encoded_dict = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"  # Return as PyTorch tensors
    )
    
    input_ids = encoded_dict["input_ids"]
    attention_mask = encoded_dict["attention_mask"]

    # Put model in evaluation mode
    model.eval()

    # Get embeddings from BERT
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    
    # Extract last hidden state (batch_size=1, seq_len, hidden_dim)
    hidden_states = outputs.hidden_states

    # Use the [CLS] token's embedding from the last layer
    cls_embedding = hidden_states[-1][:, 0, :].squeeze().cpu().numpy()

    return cls_embedding  # Shape: (hidden_dim,) e.g. (768,)

# Example Usage:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased", output_hidden_states=True)

text = "This is an example sentence."
vector = create_vector_from_text(tokenizer, model, text)
print(vector)  # Output: (768,)


[-3.77359360e-01 -3.34982514e-01 -3.20642292e-01 -4.39906210e-01
 -5.01397014e-01 -2.16923013e-01  4.74738896e-01  5.71138918e-01
 -8.48564133e-02 -1.06666736e-01 -3.79479527e-01 -4.84674238e-03
 -9.62600186e-02  1.43018831e-02  6.41196668e-01  4.23180759e-02
  2.14646354e-01  5.66469610e-01  2.44057387e-01 -4.97501552e-01
  2.44320601e-01 -3.14713627e-01 -1.83872491e-01 -4.34330940e-01
  3.69996607e-01 -4.88474965e-01 -1.60260051e-01 -5.30239880e-01
 -2.55357563e-01  2.52796188e-02  1.14062071e-01  3.28302026e-01
 -3.37487668e-01 -1.74548030e-01  5.42107224e-01 -2.44187742e-01
  2.18429476e-01 -5.26539870e-02  5.42507648e-01  2.21359476e-01
 -2.60100633e-01 -6.98961467e-02  4.54150617e-01  1.44980341e-01
  7.11286366e-02 -1.93159938e-01 -2.88296938e+00 -4.78262156e-01
 -4.72730875e-01 -2.87774324e-01 -3.70462298e-01 -8.15530196e-02
  7.47938275e-01  4.68319923e-01 -2.21923172e-01  3.83136392e-01
 -3.83444488e-01  3.79058212e-01  2.10735872e-01  1.02007911e-02
  3.55581701e-01  4.19319

In [28]:
def create_vector_from_text(tokenizer, model, text, MAX_LEN=512):
    encoded_dict = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt" 
    )
    input_ids = encoded_dict["input_ids"]
    attention_mask = encoded_dict["attention_mask"]

    model.eval()
    # Get embeddings from BERT
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
    hidden_states = outputs.hidden_states
    cls_embedding = hidden_states[-1][:, 0, :].squeeze().cpu().numpy()

    return cls_embedding

def is_plagiarism(similarity_score, threshold=0.8):
    return similarity_score >= threshold


def run_plagiarism_analysis(query_text, data, plagiarism_threshold=0.8):
    top_N=3
    # Run similarity Search
    data["similarity"] = data["vectors"].apply(lambda x: cosine_similarity(vector.reshape(1, -1), x.reshape(1, -1)))
    data["similarity"] = data["similarity"].apply(lambda x: x[0][0])

    similar_articles = data.sort_values(by='similarity', ascending=False)[0:top_N+1]
    formated_result = similar_articles[["text", "title","url", "similarity"]].reset_index(drop = True)

    similarity_score = formated_result.iloc[0]["similarity"] 
    most_similar_article = formated_result.iloc[0]["text"] 
    is_plagiarism_bool = is_plagiarism(similarity_score, plagiarism_threshold)

    plagiarism_decision = {'similarity_score': similarity_score, 
                             'is_plagiarism': is_plagiarism_bool,
                             'most_similar_article': most_similar_article, 
                             'article_submitted': query_text
                            }

    return plagiarism_decision


In [10]:
# Example Usage:
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased", output_hidden_states=True)
    
    text = "Humans (Homo sapiens) or modern humans are the most common and widespread species of primate, and the last surviving species of the genus Homo."
    vector = create_vector_from_text(tokenizer, model, text)
    print(vector)  # Output: (768,)

[-3.34091246e-01  1.72108173e-01 -4.84860092e-01  9.18453410e-02
 -3.31431329e-01  3.98206890e-01  1.10738826e+00  1.13359582e+00
 -3.88914555e-01 -2.18468040e-01  1.92202404e-01 -2.76707351e-01
 -1.09109426e+00  7.84903944e-01 -2.83685088e-01 -6.93295598e-02
 -5.73645771e-01  4.26531702e-01  3.98950547e-01 -1.98533952e-01
 -2.05863029e-01 -7.34285891e-01  3.05386111e-02  1.89474702e-01
  1.66059464e-01 -5.12983859e-01  2.48877436e-01  1.42340600e-01
  1.02755940e+00 -1.16629571e-01 -6.86427653e-01  2.65563339e-01
 -3.15122008e-01 -2.07950205e-01  3.98636878e-01 -5.55173576e-01
 -4.15504932e-01 -3.77732635e-01 -8.65861103e-02  7.23332882e-01
 -2.88181365e-01 -1.87774405e-01 -3.80989492e-01  3.01175684e-01
 -2.50935584e-01  2.91956127e-01 -2.92230177e+00  5.74320674e-01
 -3.42930198e-01  2.31188908e-02 -9.12993252e-02 -9.21960250e-02
 -6.70335591e-02  7.11991727e-01  4.92904454e-01 -5.15098311e-03
 -1.59764394e-01  4.79604274e-01 -5.89007199e-01  1.70179337e-01
 -5.75399816e-01  9.01527

In [29]:
analysis_result = run_plagiarism_analysis(vector, data, plagiarism_threshold=0.8)

In [30]:
analysis_result 

{'similarity_score': 0.8446413,
 'is_plagiarism': True,
 'most_similar_article': 'Animals (also called Metazoa) are multicellular, eukaryotic organisms in the biological kingdom Animalia. With few exceptions, animals consume organic material, breathe oxygen, are able to move, can reproduce sexually, and go through an ontogenetic stage in which their body consists of a hollow sphere of cells, the blastula, during embryonic development. Over 1.5 million living animal species have been described—of which around 1 million are insects—but it has been estimated there are over 7 million animal species in total. Animals range in length from  to . They have complex interactions with each other and their environments, forming intricate food webs. The scientific study of animals is known as zoology.\n\nMost living animal species are in Bilateria, a clade whose members have a bilaterally symmetric body plan. The Bilateria include the protostomes, containing invertebrates such as nematodes, arthrop

In [49]:
import pandas as pd
import numpy as np
def create_vector_from_text(tokenizer, model, text, MAX_LEN=512):
    # Tokenize and pad input
    encoded_dict = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"  # Return as PyTorch tensors
    )
    
    input_ids = encoded_dict["input_ids"]
    attention_mask = encoded_dict["attention_mask"]

    # Put model in evaluation mode
    model.eval()

    # Get embeddings from BERT
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    
    # Extract last hidden state (batch_size=1, seq_len, hidden_dim)
    hidden_states = outputs.hidden_states

    # Use the [CLS] token's embedding from the last layer
    cls_embedding = hidden_states[-1][:, 0, :].squeeze().cpu().numpy()

    return cls_embedding  # Shape: (hidden_dim,) e.g. (768,)

def is_plagiarism(similarity_score, threshold=0.8):
    return similarity_score >= threshold

def run_plagiarism_analysis(query_text, data, plagiarism_threshold=0.8):
    top_N = 3
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased", output_hidden_states=True)
    
    vector = create_vector_from_text(tokenizer, model, query_text)
    # Run similarity Search
    data["similarity"] = data["vectors"].apply(lambda x: cosine_similarity(vector.reshape(1, -1), x.reshape(1, -1)))
    data["similarity"] = data["similarity"].apply(lambda x: x[0][0])

    similar_articles = data.sort_values(by='similarity', ascending=False)[0:top_N+1]
    formated_result = similar_articles[["text", "title", "url", "similarity"]].reset_index(drop=True)

    similarity_score = formated_result.iloc[0]["similarity"]
    most_similar_article = formated_result.iloc[0]["text"]
    most_similar_article_url = formated_result.iloc[0]["url"]
    is_plagiarism_bool = is_plagiarism(similarity_score, plagiarism_threshold)

    plagiarism_decision = {'similarity_score': similarity_score,
                           'is_plagiarism': is_plagiarism_bool,
                           'most_similar_article': most_similar_article,
                           'most_similar_article_url': most_similar_article_url,
                           'article_submitted': query_text
                          }

    return plagiarism_decision

if __name__ == "__main__":

    query_text = "Phenyl phosphorodiamidate is an organophosphor"
    result = run_plagiarism_analysis(query_text, data)
    print(result)

{'similarity_score': 0.88794166, 'is_plagiarism': True, 'most_similar_article': 'Bromocyclopentane is an alkyl halide with the chemical formula C5H9Br.  It is a colorless to light yellow liquid at standard temperature and pressure.\n\nBromocyclopentane is a building block used in the synthesis of filaminast.\n\nBromocyclopentane is reacted with magnesium turnings in dry tetrahydrofuran making cyclopentyl Grignard reagent, a main precursor in the synthesis of Ketamine. \n\nOrganobromides\nCyclopentyl compounds', 'most_similar_article_url': 'https://en.wikipedia.org/wiki/Bromocyclopentane', 'article_submitted': 'Phenyl phosphorodiamidate is an organophosphor'}


In [54]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

def create_vector_from_text(tokenizer, model, text, MAX_LEN=512):
    # Tokenize and pad input
    encoded_dict = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"  # Return as PyTorch tensors
    )
    
    input_ids = encoded_dict["input_ids"]
    attention_mask = encoded_dict["attention_mask"]

    # Put model in evaluation mode
    model.eval()

    # Get embeddings from BERT
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    
    # Extract last hidden state (batch_size=1, seq_len, hidden_dim)
    hidden_states = outputs.hidden_states

    # Use the [CLS] token's embedding from the last layer
    cls_embedding = hidden_states[-1][:, 0, :].squeeze().cpu().numpy()

    return cls_embedding  # Shape: (hidden_dim,) e.g. (768,)

def is_plagiarism(similarity_score, threshold=0.8):
    return similarity_score >= threshold

def run_plagiarism_analysis(query_text, data, plagiarism_threshold=0.8):
    top_N = 3
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased", output_hidden_states=True)
    
    vector = create_vector_from_text(tokenizer, model, query_text)
    # Run similarity search
    data["similarity"] = data["vectors"].apply(lambda x: cosine_similarity(vector.reshape(1, -1), np.array(x).reshape(1, -1)))
    data["similarity"] = data["similarity"].apply(lambda x: x[0][0])

    similar_articles = data.sort_values(by='similarity', ascending=False).head(top_N)
    formated_result = similar_articles[["text", "title", "url", "similarity"]].reset_index(drop=True)

    similarity_score = formated_result.iloc[0]["similarity"]
    most_similar_article = formated_result.iloc[0]["text"]
    most_similar_article_url = formated_result.iloc[0]["url"]
    is_plagiarism_bool = is_plagiarism(similarity_score, plagiarism_threshold)

    plagiarism_decision = {'similarity_score': similarity_score,
                           'is_plagiarism': is_plagiarism_bool,
                           'most_similar_article': most_similar_article,
                           'most_similar_article_url': most_similar_article_url,
                           'article_submitted': query_text
                          }

    return plagiarism_decision

if __name__ == "__main__":
    
    query_text = "Machine learning is a subfield of an AI."
    result = run_plagiarism_analysis(query_text, data)
    print(result)

{'similarity_score': 0.8786567, 'is_plagiarism': True, 'most_similar_article': 'Telexistence is fundamentally a concept named for the general technology that enables a human being to have a real-time sensation of being at a place other than where they actually exist, and being able to interact with the remote environment, which may be real, virtual, or a combination of both. It also refers to an advanced type of teleoperation system that enables an operator at the control to perform remote tasks dexterously with the feeling of existing in a surrogate robot working in a remote environment. Telexistence in the real environment through a virtual environment is also possible. This concept was first proposed by Susumu Tachi in Japan in 1980 and 1981 as patents and the first report was published in Japanese in 1982 and in English in 1984.\n\nSee also\n Virtual reality\n Methods of virtual reality\n Virtuality Continuum\n Telepresence\n\nReferences\n\nVirtual reality\nReality\nUser interface 

In [57]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Ensure you have the necessary NLTK data files
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_text(text, remove_stopwords=True, lemmatize=True):
    """Preprocess the text by removing stopwords and lemmatizing."""
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        text = ' '.join([word for word in text.split() if word.lower() not in stop_words])
    
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
    
    return text

def calculate_similarity(doc1, doc2, method='tfidf'):
    """Calculate the similarity between two documents."""
    if method == 'tfidf':
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform([doc1, doc2])
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
        return similarity[0][0]
    else:
        raise ValueError("Unsupported method. Use 'tfidf'.")

def compare_multiple_documents(input_vector, dataset_vectors):
    results = []
    
    # Ensure the input vector is a 2D array
    input_vector = input_vector.reshape(1, -1)
    
    for idx, row_vector in enumerate(dataset_vectors, 1):
        # Ensure the row vector is a 2D array
        row_vector = np.array(row_vector).reshape(1, -1)
        
        # Calculate cosine similarity
        similarity = cosine_similarity(input_vector, row_vector)[0][0]
        
        # Append the result
        results.append({
            "doc_id": idx,  # Row index in the dataset
            "similarity": similarity * 100,  # Convert to percentage
            "input_vector": input_vector.flatten().tolist(),  # Flatten for readability
            "row_vector": row_vector.flatten().tolist()  # Flatten for readability
        })
    
    return results

def create_vector_from_text(tokenizer, model, text, MAX_LEN=512):
    # Tokenize and pad input
    encoded_dict = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"  # Return as PyTorch tensors
    )
    
    input_ids = encoded_dict["input_ids"]
    attention_mask = encoded_dict["attention_mask"]

    # Put model in evaluation mode
    model.eval()

    # Get embeddings from BERT
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    
    # Extract last hidden state (batch_size=1, seq_len, hidden_dim)
    hidden_states = outputs.hidden_states

    # Use the [CLS] token's embedding from the last layer
    cls_embedding = hidden_states[-1][:, 0, :].squeeze().cpu().numpy()

    return cls_embedding  # Shape: (hidden_dim,) e.g. (768,)

def is_plagiarism(similarity_score, threshold=0.8):
    return similarity_score >= threshold

def run_plagiarism_analysis(query_text, data, plagiarism_threshold=0.8):
    top_N = 3
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased", output_hidden_states=True)
    
    vector = create_vector_from_text(tokenizer, model, query_text)
    # Run similarity search
    data["similarity"] = data["vectors"].apply(lambda x: cosine_similarity(vector.reshape(1, -1), np.array(x).reshape(1, -1)))
    data["similarity"] = data["similarity"].apply(lambda x: x[0][0])

    similar_articles = data.sort_values(by='similarity', ascending=False).head(top_N)
    formated_result = similar_articles[["text", "title", "url", "similarity"]].reset_index(drop=True)

    similarity_score = formated_result.iloc[0]["similarity"]
    most_similar_article = formated_result.iloc[0]["text"]
    most_similar_article_url = formated_result.iloc[0]["url"]
    is_plagiarism_bool = is_plagiarism(similarity_score, plagiarism_threshold)

    plagiarism_decision = {'similarity_score': similarity_score,
                           'is_plagiarism': is_plagiarism_bool,
                           'most_similar_article': most_similar_article,
                           'most_similar_article_url': most_similar_article_url,
                           'article_submitted': query_text
                          }

    return plagiarism_decision

if __name__ == "__main__":
    # Example usage of compare_multiple_documents
    documents = [
        "Machine learning is a subfield of AI.",
        "AI includes various subfields like machine learning.",
        "Deep learning is a subset of machine learning."
    ]
    comparison_results = compare_multiple_documents(documents)
    print("Comparison Results:")
    for result in comparison_results:
        print(result)

    # Example usage of run_plagiarism_analysis
    query_text = "Machine learning is a subfield of AI."
    result = run_plagiarism_analysis(query_text, data)
    print("\nPlagiarism Analysis Result:")
    print(result)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Comparison Results:
{'doc1_id': 1, 'doc2_id': 2, 'similarity': 40.73526042885675, 'doc1_processed': 'Machine learning subfield AI.', 'doc2_processed': 'AI includes various subfields like machine learning.'}
{'doc1_id': 1, 'doc2_id': 3, 'similarity': 41.10653709825173, 'doc1_processed': 'Machine learning subfield AI.', 'doc2_processed': 'Deep learning subset machine learning.'}
{'doc1_id': 2, 'doc2_id': 3, 'similarity': 29.096881261556007, 'doc1_processed': 'AI includes various subfields like machine learning.', 'doc2_processed': 'Deep learning subset machine learning.'}

Plagiarism Analysis Result:
{'similarity_score': 0.8554057, 'is_plagiarism': True, 'most_similar_article': 'The smart cow problem is the concept that, when a group of individuals is faced with a technically difficult task, only one of their members has to solve it. When the problem has been solved once, an easily repeatable method may be developed, allowing the less technically proficient members of the group to accomp

## WOrking code to use in final model >>>>

In [25]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer, util
import torch
from transformers import BertTokenizer, BertModel
import re

# Load Sentence-BERT model
sbert_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Load BERT tokenizer & model (optional if needed for a different vectorization approach)
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Function to create embeddings using SBERT
def create_vector_from_text(text):
    inputs = bert_tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    
    with torch.no_grad():
        outputs = bert_model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # Extract [CLS] token embedding
    
    return cls_embedding.numpy().flatten()  # Convert to NumPy array

# Function to compute exact match
def exact_match(query_text, candidate_text):
    query_cleaned = re.sub(r'\W+', ' ', query_text.lower())
    candidate_cleaned = re.sub(r'\W+', ' ', candidate_text.lower())
    return query_cleaned in candidate_cleaned or candidate_cleaned in query_cleaned

# Jaccard Similarity Function
def jaccard_similarity(query_text, candidate_text):
    query_set = set(query_text.lower().split())
    candidate_set = set(candidate_text.lower().split())
    
    intersection = len(query_set.intersection(candidate_set))
    union = len(query_set.union(candidate_set))
    
    return intersection / union if union != 0 else 0

# Function to compute semantic similarity using SBERT
def semantic_similarity(query_text, candidate_text):
    query_embedding = sbert_model.encode([query_text])
    candidate_embedding = sbert_model.encode([candidate_text])
    return cosine_similarity(query_embedding, candidate_embedding)[0][0]

def get_flattened_vector(row):
    return np.array(row['vectors']).flatten()

# Function to check similarity using SBERT & Jaccard
def check_similarity_and_plagiarism(query_text, data, cosine_threshold=0.8, jaccard_threshold=0.8):
    query_vector = create_vector_from_text(query_text)  # Get vector for the query
    similarities = []

    for idx, row in data.iterrows():
        row_vector = get_flattened_vector(row)  # Flatten the vector for the dataset row
        
        # Ensure both vectors have the same dimensionality (768)
        if query_vector.shape[0] != row_vector.shape[0]:
            raise ValueError(f"Dimension mismatch: query_vector has shape {query_vector.shape}, row_vector has shape {row_vector.shape}")
        
        # Normalize both vectors
        query_vector_normalized = normalize(query_vector.reshape(1, -1))[0]
        row_vector_normalized = normalize(row_vector.reshape(1, -1))[0]
        
        # Calculate cosine similarity
        cosine_sim = cosine_similarity(query_vector_normalized.reshape(1, -1), row_vector_normalized.reshape(1, -1))[0][0]
        
        # Check if the similarity exceeds the threshold for cosine similarity
        is_similar = cosine_sim >= cosine_threshold
        
        # Check for plagiarism (exact match or significant overlap) using Jaccard Similarity
        jaccard_sim = jaccard_similarity(query_text, row['text'])
        is_plagiarized = (cosine_sim >= cosine_threshold and jaccard_sim >= jaccard_threshold) or exact_match(query_text, row['text'])
        
        similarities.append({
            'text': row['text'],
            'title': row['title'],
            'similarity': cosine_sim,
            'jaccard_similarity': jaccard_sim,
            'is_similar': is_similar,
            'is_plagiarized': is_plagiarized,
            'url': row['url'],
        })

    # Sort by similarity (highest cosine similarity first)
    similarities.sort(key=lambda x: x['similarity'], reverse=True)
    
    # Return top N most similar, with plagiarism checks
    return similarities[:5]

# Example query
query_text = "Long-acting reversible contraceptives (LARC) are methods of birth control that provide effective contraception for an extended period without requiring user action. They include hormonal and non-hormonal intrauterine devices (IUDs) and subdermal hormonal contraceptive implants. They are the most effective reversible methods of contraception because their efficacy is not reliant on patient compliance. The failure rates of IUDs and implants is less than 1% per year."

# Get the top similar rows based on cosine similarity and plagiarism check
top_similar_rows = check_similarity_and_plagiarism(query_text, data, cosine_threshold=0.8, jaccard_threshold=0.8)

# Output top similar rows
print("Top similar rows:")
for row in top_similar_rows:
    print(row)

# Compute semantic similarity among the top results
top_semantic_match = None
highest_semantic_similarity = -1

for row in top_similar_rows:
    semantic_sim = semantic_similarity(query_text, row['text'])
    if semantic_sim > highest_semantic_similarity:
        highest_semantic_similarity = semantic_sim
        top_semantic_match = row

# Output the most semantically similar text
if top_semantic_match:
    print(f"\nMost semantically similar text:\n")
    print(f"Title: {top_semantic_match['title']}")
    print(f"URL: {top_semantic_match['url']}")
    print(f"Cosine Similarity: {top_semantic_match['similarity']:.2f}")
    print(f"Semantic Similarity: {highest_semantic_similarity:.2f}")
    print(f"Plagiarism: {'Yes' if top_semantic_match['is_plagiarized'] else 'No'}")

# Compute and display the most semantically similar row using Sentence-BERT
query_embedding = sbert_model.encode(query_text, convert_to_tensor=True)

# Loop through each top similar row and compute cosine similarity
similarities = []
for row in top_similar_rows:
    candidate_text = row['text']
    
    # Compute the embedding for the candidate text
    row_embedding = sbert_model.encode(candidate_text, convert_to_tensor=True)
    
    # Compute cosine similarity between the query and candidate
    similarity = util.cos_sim(query_embedding, row_embedding)[0][0]
    similarities.append((row['title'], candidate_text, similarity.item()))  # Store similarity as a tuple (title, text, similarity)

# Rank candidates by similarity (descending order)
similarities.sort(key=lambda x: x[2], reverse=True)

# Print results
print("\nRanked candidates by similarity:")
for title, text, similarity in similarities:
    print(f"Title: {title}, Similarity: {similarity:.4f}")


Top similar rows:
{'text': 'Biologics for immunosuppression (often called "biologics" or "biological therapy" or "biological drugs") are a class of immunosuppressive drugs which chemically are biopharmaceutical treatments.\n\nBiologics treat medical conditions where immunotherapy is effective. Some conditions which biologics treat are psoriatic arthritis, rheumatism, and inflammatory bowel disease.\n\nExample drugs in this category include adalimumab, certolizumab, etanercept, golimumab, infliximab, belimumab, and ustekinumab.\n\nMedical uses\nBiologics provide immunotherapy. For people with moderate to severe psoriatic arthritis, biologics can provide some relief of the symptoms.\n\nBiologics can function as disease-modifying antirheumatic drugs. They can treat inflammatory bowel disease.\n\nContraindications\nBiologics should be used after considering other less invasive treatments. Before using biologics to treat psoriasis, easier treatments of moisturizer or light therapy can provi


Most semantically similar text:

Title: Long-acting reversible contraception
URL: https://en.wikipedia.org/wiki/Long-acting%20reversible%20contraception
Cosine Similarity: 0.87
Semantic Similarity: 0.97
Plagiarism: No

Ranked candidates by similarity:
Title: Long-acting reversible contraception, Similarity: 0.9679
Title: Vibroejaculation, Similarity: 0.3249
Title: Antihypertensive drug, Similarity: 0.2845
Title: Anti-aging supplements, Similarity: 0.2520
Title: Biologics for immunosuppression, Similarity: 0.0797
